# Width-QC candidate-model review

This notebook reviews the versioned Phase 4 dataset and Phase 5 leave-one-example-out model artifacts. It performs no hidden training-data preparation and does not authorize or deploy a production model. Rebuild the inputs with `python -m network_variants.width_review_dataset` and `python -m network_variants.width_qc_model`.

In [ ]:
import os
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-width-qc-review')

import hashlib
import json
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import Markdown, display
from sklearn.metrics import precision_recall_curve, roc_curve

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'network_variants').is_dir() and (candidate / 'rivgraph_centerline').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the river-hierarchy repository root.')

REPO_ROOT = find_repo_root(Path.cwd())
DATASET_DIR = REPO_ROOT / 'network_variants' / 'output' / 'width_qc_dataset'
MODEL_DIR = REPO_ROOT / 'network_variants' / 'output' / 'width_qc_model'
RUNS_DIR = REPO_ROOT / 'rivgraph_centerline' / 'outputs' / 'runs'
REPO_ROOT

## Load and verify versioned artifacts

In [ ]:
dataset_path = DATASET_DIR / 'width_qc_training_data_v1.csv'
dataset_report_path = DATASET_DIR / 'width_qc_training_report_v1.json'
dataset_manifest_path = DATASET_DIR / 'width_qc_training_manifest_v1.json'
predictions_path = MODEL_DIR / 'width_qc_loeo_predictions_v1.csv'
per_example_path = MODEL_DIR / 'width_qc_loeo_per_example_metrics_v1.csv'
thresholds_path = MODEL_DIR / 'width_qc_threshold_metrics_v1.csv'
calibration_path = MODEL_DIR / 'width_qc_calibration_v1.csv'
model_summary_path = MODEL_DIR / 'width_qc_model_validation_v1.json'
model_manifest_path = MODEL_DIR / 'width_qc_model_manifest_v1.json'

dataset = pd.read_csv(dataset_path, dtype={'example_id': 'string'})
predictions = pd.read_csv(predictions_path, dtype={'example_id': 'string'})
per_example = pd.read_csv(per_example_path, dtype={'held_out_example': 'string'})
thresholds = pd.read_csv(thresholds_path)
calibration = pd.read_csv(calibration_path)
dataset_report = json.loads(dataset_report_path.read_text())
dataset_manifest = json.loads(dataset_manifest_path.read_text())
model_summary = json.loads(model_summary_path.read_text())
model_manifest = json.loads(model_manifest_path.read_text())

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

assert sha256(dataset_path) == dataset_manifest['dataset_sha256']
assert model_manifest['dataset_sha256'] == dataset_manifest['dataset_sha256']
assert model_summary['production_authorized'] is False
trained_sklearn = model_summary['software_versions']['scikit_learn']
runtime_sklearn = sklearn.__version__
if runtime_sklearn != trained_sklearn:
    display(Markdown(
        f"⚠️ **scikit-learn version mismatch:** candidate bundles were trained with `{trained_sklearn}`, "
        f"but this kernel uses `{runtime_sklearn}`. Joblib model loading is intentionally avoided in this "
        "notebook; diagnostic surfaces below are reconstructed from the portable, versioned coefficients. "
        "Use the `river-hierarchy-rivgraph` kernel for direct bundle inference."
    ))
display(pd.DataFrame([{
    'dataset_id': dataset_manifest['dataset_id'],
    'rows': len(dataset),
    'examples': dataset['example_id'].nunique(),
    'candidate_models': predictions['width_model_name'].nunique(),
    'production_authorized': model_summary['production_authorized'],
}]))

## Manual-review counts and missingness

In [ ]:
review_counts = dataset.groupby(['example_id', 'manual_decision']).size().unstack(fill_value=0)
display(review_counts)
review_counts.plot(kind='bar', stacked=True, figsize=(12, 4), color={'good': '#4C78A8', 'bad': '#E45756'})
plt.ylabel('Reviewed candidate links')
plt.title('Manual width-review labels by held-out example')
plt.tight_layout()
plt.show()

missing = pd.DataFrame(dataset_report['missing_values']).T.sort_values('missing_rows', ascending=False)
display(missing)

## Predictor distributions by manual decision

In [ ]:
features = model_summary['feature_columns']
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.ravel()
for ax, feature in zip(axes, features):
    for decision, color in [('good', '#4C78A8'), ('bad', '#E45756')]:
        values = dataset.loc[dataset['manual_decision'].eq(decision), feature].dropna()
        ax.hist(values, bins=30, density=True, alpha=0.5, label=decision, color=color)
    ax.set_title(feature.replace('_', ' '))
    ax.legend()
for ax in axes[len(features):]:
    ax.axis('off')
fig.suptitle('Agreed model predictors within the reviewed candidate population', y=1.01)
fig.tight_layout()
plt.show()

## Pairwise relationships among all classifier and interpretation variables

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

variable_labels = {
    'length_over_raw_wet_width': 'length / wet width',
    'split_sensitive': 'split sensitive',
    'opposition_cos_trimmed_mean': 'opposition cos mean',
    'opposition_cos_trimmed_median': 'opposition cos median',
    'invalid_sample_fraction': 'invalid sample fraction',
    'upstream_continuity_ratio': 'upstream ratio',
    'upstream_continuity_abs_deviation': 'upstream abs deviation',
    'raw_wet_width': 'raw wet width',
    'raw_total_width': 'raw total width',
    'wet_fraction': 'wet fraction',
    'downstream_continuity_abs_deviation': 'downstream abs deviation',
    'mean_bank_chord_difference': 'bank chord difference',
    'mean_relative_dt_difference': 'relative DT difference',
    'n_wet_threads_mean': 'mean wet threads',
    'link_length': 'link length',
}

def partial_logistic_boundary(x_variable, y_variable, x_limits, y_limits, *, reference_frame, threshold):
    if x_variable not in features or y_variable not in features:
        return None
    diagnostics = model_summary['candidate_models']['logistic_regression']['full_fit_diagnostics']
    coefficients = pd.Series(diagnostics['standardized_coefficients'], dtype=float)
    means = pd.Series(diagnostics['standardization_means'], dtype=float)
    scales = pd.Series(diagnostics['standardization_scales'], dtype=float)
    reference = reference_frame[features].median(numeric_only=True)
    score_target = float(np.log(np.clip(threshold, 1e-9, 1 - 1e-9) / np.clip(1 - threshold, 1e-9, 1)))
    fixed_score = float(diagnostics['intercept'])
    for variable in features:
        if variable not in (x_variable, y_variable):
            fixed_score += coefficients[variable] * (reference[variable] - means[variable]) / scales[variable]
    beta_x = coefficients[x_variable] / scales[x_variable]
    beta_y = coefficients[y_variable] / scales[y_variable]
    centered_target = score_target - fixed_score + beta_x * means[x_variable] + beta_y * means[y_variable]
    if abs(beta_y) > 1e-12:
        line_x = np.linspace(x_limits[0], x_limits[1], 160)
        line_y = (centered_target - beta_x * line_x) / beta_y
        visible = (line_y >= y_limits[0]) & (line_y <= y_limits[1])
        return line_x[visible], line_y[visible]
    if abs(beta_x) > 1e-12:
        line_y = np.linspace(y_limits[0], y_limits[1], 160)
        line_x = np.full_like(line_y, centered_target / beta_x)
        visible = (line_x >= x_limits[0]) & (line_x <= x_limits[1])
        return line_x[visible], line_y[visible]
    return None

def pairwise_review_matrix(
    frame, variables, *, group_column, palette, group_labels, title,
    decision_boundary=False, decision_reference_frame=None, decision_threshold=0.5,
):
    values = frame[['example_id', 'id_link', *variables, group_column]].copy()
    for variable in variables:
        values[variable] = pd.to_numeric(values[variable], errors='coerce')
    n = len(variables)
    fig = make_subplots(rows=n, cols=n, horizontal_spacing=0.006, vertical_spacing=0.006)
    limits = {}
    for variable in variables:
        finite = values[variable].dropna()
        low, high = finite.quantile([0.01, 0.99]) if len(finite) else (0.0, 1.0)
        if not np.isfinite(low) or not np.isfinite(high) or low == high:
            center = float(finite.median()) if len(finite) else 0.0
            low, high = center - 0.5, center + 0.5
        padding = 0.04 * (high - low)
        limits[variable] = (float(low - padding), float(high + padding))

    ordered_groups = [group for group in palette if group in set(values[group_column].dropna())]
    for row_index, y_variable in enumerate(variables):
        for column_index, x_variable in enumerate(variables):
            if row_index == column_index:
                for group in ordered_groups:
                    series = values.loc[values[group_column].eq(group), x_variable].dropna()
                    fig.add_trace(
                        go.Histogram(
                            x=series, nbinsx=22, histnorm='probability density', opacity=0.48,
                            marker_color=palette[group], name=group_labels[group],
                            legendgroup=str(group), showlegend=(row_index == 0 and column_index == 0),
                            hoverinfo='skip',
                        ),
                        row=row_index + 1, col=column_index + 1,
                    )
                fig.update_xaxes(range=limits[x_variable], row=row_index + 1, col=column_index + 1)
            elif row_index > column_index:
                for group in ordered_groups:
                    subset = values.loc[values[group_column].eq(group)]
                    customdata = np.column_stack([subset['example_id'].astype(str), subset['id_link'].astype(str)])
                    fig.add_trace(
                        go.Scattergl(
                            x=subset[x_variable], y=subset[y_variable], mode='markers',
                            marker={'size': 6, 'opacity': 0.55, 'color': palette[group]},
                            customdata=customdata,
                            hovertemplate='network %{customdata[0]}<br>link %{customdata[1]}<extra></extra>',
                            name=group_labels[group], legendgroup=str(group), showlegend=False,
                        ),
                        row=row_index + 1, col=column_index + 1,
                    )
                if decision_boundary:
                    boundary = partial_logistic_boundary(
                        x_variable, y_variable, limits[x_variable], limits[y_variable],
                        reference_frame=decision_reference_frame, threshold=decision_threshold,
                    )
                    if boundary is not None and len(boundary[0]):
                        fig.add_trace(
                            go.Scatter(
                                x=boundary[0], y=boundary[1], mode='lines',
                                line={'color': 'black', 'width': 2, 'dash': 'dash'},
                                name=f'Partial logistic boundary (p={decision_threshold:.2f})',
                                legendgroup='decision_boundary',
                                showlegend=(row_index == 1 and column_index == 0), hoverinfo='skip',
                            ),
                            row=row_index + 1, col=column_index + 1,
                        )
                fig.update_xaxes(range=limits[x_variable], row=row_index + 1, col=column_index + 1)
                fig.update_yaxes(range=limits[y_variable], row=row_index + 1, col=column_index + 1)
            else:
                correlation_value = values[[x_variable, y_variable]].corr().iloc[0, 1]
                text = f'r = {correlation_value:.2f}' if np.isfinite(correlation_value) else 'r = NA'
                fig.add_annotation(
                    text=text, x=0.5, y=0.5, xref='x domain', yref='y domain',
                    showarrow=False, font={'size': 13}, row=row_index + 1, col=column_index + 1,
                )
                fig.update_xaxes(visible=False, row=row_index + 1, col=column_index + 1)
                fig.update_yaxes(visible=False, row=row_index + 1, col=column_index + 1)

            if row_index == n - 1:
                fig.update_xaxes(
                    title_text=variable_labels.get(x_variable, x_variable), title_font={'size': 10},
                    tickfont={'size': 8}, row=row_index + 1, col=column_index + 1,
                )
            elif row_index >= column_index:
                fig.update_xaxes(showticklabels=False, row=row_index + 1, col=column_index + 1)
            if column_index == 0:
                fig.update_yaxes(
                    title_text=variable_labels.get(y_variable, y_variable), title_font={'size': 10},
                    tickfont={'size': 8}, row=row_index + 1, col=column_index + 1,
                )
            elif row_index >= column_index:
                fig.update_yaxes(showticklabels=False, row=row_index + 1, col=column_index + 1)

    subtitle = (
        '<br><sup>Black dashed lines are partial logistic boundaries; other classifier inputs are fixed at medians.</sup>'
        if decision_boundary else ''
    )
    fig.update_layout(
        title={'text': title + subtitle, 'x': 0.5},
        width=max(1000, 245 * n), height=max(1000, 245 * n),
        barmode='overlay', template='plotly_white', hovermode='closest',
        legend={'orientation': 'h', 'y': 1.035, 'x': 0.5, 'xanchor': 'center'},
        margin={'l': 90, 'r': 30, 't': 115, 'b': 90},
    )
    fig.show()

classification_columns = list(features)
pairwise_review_matrix(
    dataset, classification_columns,
    group_column='manual_decision',
    palette={'good': '#4C78A8', 'bad': '#E45756'},
    group_labels={'good': 'Manual good', 'bad': 'Manual bad'},
    title='All classifier inputs against each other — complete reviewed population',
    decision_boundary=True, decision_reference_frame=dataset,
    decision_threshold=float(model_summary['candidate_models']['logistic_regression']['recommended_threshold']),
)

## Correlation and collinearity review

In [ ]:
correlation = dataset[features].corr()
fig, ax = plt.subplots(figsize=(9, 7))
image = ax.imshow(correlation, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(len(features)), [f.replace('_', '\n') for f in features], rotation=45, ha='right')
ax.set_yticks(range(len(features)), [f.replace('_', ' ') for f in features])
for i in range(len(features)):
    for j in range(len(features)):
        ax.text(j, i, f'{correlation.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(image, ax=ax, label='Pearson correlation')
ax.set_title('Predictor correlation matrix')
fig.tight_layout()
plt.show()
display(correlation)

## Logistic coefficients, odds ratios, and linear-SVM weights

In [ ]:
log_diag = model_summary['candidate_models']['logistic_regression']['full_fit_diagnostics']
linear_diag = model_summary['candidate_models']['linear_svm']['full_fit_diagnostics']
coefficient_table = pd.DataFrame({
    'logistic_standardized_coefficient': log_diag['standardized_coefficients'],
    'logistic_standardized_odds_ratio': log_diag['standardized_odds_ratios'],
    'linear_svm_standardized_weight': linear_diag['standardized_coefficients'],
}).sort_values('logistic_standardized_coefficient', key=lambda values: values.abs(), ascending=False)
display(coefficient_table)
coefficient_table[['logistic_standardized_coefficient', 'linear_svm_standardized_weight']].plot.barh(figsize=(9, 5))
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Full-fit standardized linear effects (diagnostic only)')
plt.tight_layout()
plt.show()

## Candidate-model comparison and per-example performance

In [ ]:
model_rows = []
for model_name, details in model_summary['candidate_models'].items():
    model_rows.append({'model_name': model_name, 'recommended_threshold': details['recommended_threshold'], **details['metrics']})
model_comparison = pd.DataFrame(model_rows).set_index('model_name')
display(model_comparison[['roc_auc', 'average_precision', 'brier_score', 'precision', 'recall', 'false_correction_rate', 'split_sensitive_false_correction_rate']])

metric_plot = model_comparison[['roc_auc', 'average_precision', 'precision', 'recall']]
metric_plot.plot.bar(figsize=(10, 4), ylim=(0.85, 1.0))
plt.title('Pooled leave-one-example-out metrics')
plt.tight_layout()
plt.show()

provisional_model = model_summary['provisional_candidate_model']
provisional_by_example = per_example.loc[per_example['model_name'].eq(provisional_model)].copy()
display(provisional_by_example[['held_out_example', 'rows', 'roc_auc', 'average_precision', 'precision', 'recall', 'false_correction_rate', 'split_sensitive_false_correction_rate']])

## Held-out ROC and precision–recall curves

In [ ]:
fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(12, 5))
for model_name, group in predictions.groupby('width_model_name'):
    y = group['requires_correction'].to_numpy()
    probability = group['width_model_probability'].to_numpy()
    fpr, tpr, _ = roc_curve(y, probability)
    precision, recall, _ = precision_recall_curve(y, probability)
    metrics = model_summary['candidate_models'][model_name]['metrics']
    ax_roc.plot(fpr, tpr, label=f"{model_name} (AUC={metrics['roc_auc']:.3f})")
    ax_pr.plot(recall, precision, label=f"{model_name} (AP={metrics['average_precision']:.3f})")
ax_roc.plot([0, 1], [0, 1], '--', color='grey')
ax_roc.set(xlabel='False-positive rate', ylabel='True-positive rate', title='Held-out ROC')
ax_pr.set(xlabel='Recall', ylabel='Precision', title='Held-out precision–recall')
ax_roc.legend()
ax_pr.legend()
fig.tight_layout()
plt.show()

## Probability calibration

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for model_name, group in calibration.groupby('model_name'):
    observed = group.dropna(subset=['mean_probability', 'observed_correction_rate'])
    ax.plot(observed['mean_probability'], observed['observed_correction_rate'], marker='o', label=model_name)
ax.plot([0, 1], [0, 1], '--', color='grey', label='perfect calibration')
ax.set(xlabel='Mean held-out probability', ylabel='Observed correction fraction', title='Calibration by fixed probability bin')
ax.legend()
plt.tight_layout()
plt.show()
display(model_comparison[['brier_score']].sort_values('brier_score'))

## Threshold trade-offs and split-sensitive risk

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for model_name, group in thresholds.groupby('model_name'):
    axes[0].plot(group['threshold'], group['precision'], label=f'{model_name} precision')
    axes[0].plot(group['threshold'], group['recall'], linestyle='--', label=f'{model_name} recall')
    axes[1].plot(group['threshold'], group['split_sensitive_false_correction_rate'], label=model_name)
axes[0].set(xlabel='Threshold', ylabel='Metric', title='Precision/recall trade-off')
axes[1].axhline(0.05, color='red', linestyle='--', label='provisional 5% maximum')
axes[1].set(xlabel='Threshold', ylabel='False-correction rate', title='Manually good split-sensitive links')
axes[0].legend(fontsize=8)
axes[1].legend()
fig.tight_layout()
plt.show()

split_columns = ['recommended_threshold', 'split_sensitive_true_negative', 'split_sensitive_false_positive', 'split_sensitive_false_negative', 'split_sensitive_true_positive', 'split_sensitive_precision', 'split_sensitive_recall', 'split_sensitive_false_correction_rate']
display(model_comparison[split_columns])

## Partial decision-surface diagnostics

In [ ]:
surface_x = 'upstream_continuity_abs_deviation'
surface_y = 'opposition_cos_trimmed_mean'
x_values = np.linspace(dataset[surface_x].quantile(0.01), dataset[surface_x].quantile(0.99), 120)
y_values = np.linspace(dataset[surface_y].quantile(0.01), dataset[surface_y].quantile(0.99), 120)
xx, yy = np.meshgrid(x_values, y_values)
base = {feature: dataset[feature].median() for feature in features}
surface = pd.DataFrame({feature: np.full(xx.size, value) for feature, value in base.items()})
surface[surface_x] = xx.ravel()
surface[surface_y] = yy.ravel()

def portable_linear_surface(model_name: str, values: pd.DataFrame) -> np.ndarray:
    diagnostics = model_summary['candidate_models'][model_name]['full_fit_diagnostics']
    medians = pd.Series(diagnostics['imputation_medians'])
    means = pd.Series(diagnostics['standardization_means'])
    scales = pd.Series(diagnostics['standardization_scales'])
    coefficients = pd.Series(diagnostics['standardized_coefficients'])
    standardized = (values[features].fillna(medians) - means) / scales
    return diagnostics['intercept'] + standardized.to_numpy() @ coefficients[features].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharex=True, sharey=True)
for ax, model_name in zip(axes, ['logistic_regression', 'linear_svm']):
    score = portable_linear_surface(model_name, surface).reshape(xx.shape)
    if model_name == 'logistic_regression':
        plotted = 1.0 / (1.0 + np.exp(-np.clip(score, -40, 40)))
        levels = np.linspace(0, 1, 11)
        colorbar_label = 'Correction probability'
    else:
        plotted = score
        limit = max(float(np.nanpercentile(np.abs(score), 95)), 1e-6)
        levels = np.linspace(-limit, limit, 11)
        colorbar_label = 'Linear-SVM decision score'
    contour = ax.contourf(xx, yy, plotted, levels=levels, cmap='RdYlBu_r', alpha=0.75, extend='both')
    ax.contour(xx, yy, score, levels=[0], colors='black', linewidths=1.2)
    sample = dataset.sample(min(500, len(dataset)), random_state=0)
    ax.scatter(sample[surface_x], sample[surface_y], c=sample['requires_correction'], cmap='coolwarm', s=8, alpha=0.35, edgecolors='none')
    ax.set_title(model_name)
    ax.set_xlabel(surface_x.replace('_', ' '))
    fig.colorbar(contour, ax=ax, label=colorbar_label, shrink=0.85)
axes[0].set_ylabel(surface_y.replace('_', ' '))
fig.suptitle('Portable two-feature partial response; all other predictors fixed at medians')
fig.tight_layout()
plt.show()
display(Markdown(
    'The nonlinear RBF-SVM has no portable coefficient representation. Its held-out metrics, calibration, '
    'and threshold behavior are reviewed above; direct bundle diagnostics require the training environment.'
))

## Manual/model disagreements and routing-impact ranking

In [ ]:
classification_columns = list(features)
context_columns = [
    'raw_wet_width', 'raw_total_width', 'raw_dry_width', 'wet_fraction',
    'downstream_continuity_ratio', 'downstream_continuity_abs_deviation',
    'mean_bank_chord_difference', 'mean_relative_dt_difference', 'max_relative_dt_difference',
    'n_wet_threads_mean', 'n_wet_threads_max', 'link_length',
    'width_sample_count', 'n_invalid_width_samples', 'n_trimmed_qc_samples',
]
review_values = dataset[['example_id', 'id_link', *classification_columns, *context_columns]].copy()
selected = (
    predictions.loc[predictions['width_model_name'].eq(provisional_model)]
    .drop(columns=['split_sensitive'], errors='ignore')
    .merge(review_values, on=['example_id', 'id_link'], how='left', validate='1:1')
)
disagreements = selected.loc[selected['manual_model_disagreement']].sort_values('routing_impact_score', ascending=False)
disagreements['error_type'] = np.where(
    disagreements['requires_correction'].eq(0), 'false_correction', 'missed_correction'
)
false_positives = disagreements.loc[(disagreements['requires_correction'] == 0) & (disagreements['width_model_prediction'] == 1)]
false_negatives = disagreements.loc[(disagreements['requires_correction'] == 1) & (disagreements['width_model_prediction'] == 0)]
disagreement_table_columns = [
    'example_id', 'id_link', 'error_type', 'width_model_probability', 'width_model_threshold',
    'routing_impact_score', *classification_columns, *context_columns,
]
display(Markdown(f"**{len(disagreements)} disagreements:** {len(false_positives)} false corrections and {len(false_negatives)} missed corrections."))
display(Markdown('The **classification inputs** appear immediately after `routing_impact_score`; the remaining columns are interpretation context and were not passed to the classifier.'))
display(Markdown('### Highest-routing-impact false corrections'))
display(false_positives.loc[:, disagreement_table_columns].head(20))
display(Markdown('### Highest-routing-impact missed corrections'))
display(false_negatives.loc[:, disagreement_table_columns].head(20))

## Disagreement predictor plots

In [ ]:
error_colors = {'false_correction': '#E66101', 'missed_correction': '#0571B0'}
error_labels = {'false_correction': 'False correction (manual good)', 'missed_correction': 'Missed correction (manual bad)'}
median_width = max(float(disagreements['raw_wet_width'].median()), 1e-9)
point_sizes = (45.0 * np.sqrt(disagreements['raw_wet_width'].clip(lower=0) / median_width)).clip(20, 260)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for error_type, group in disagreements.groupby('error_type'):
    sizes = point_sizes.loc[group.index]
    axes[0].scatter(
        group['upstream_continuity_abs_deviation'], group['opposition_cos_trimmed_mean'],
        s=sizes, color=error_colors[error_type], alpha=0.72, edgecolor='white', linewidth=0.6,
        label=error_labels[error_type],
    )
    axes[1].scatter(
        group['length_over_raw_wet_width'], group['width_model_probability'],
        s=sizes, color=error_colors[error_type], alpha=0.72, edgecolor='white', linewidth=0.6,
        label=error_labels[error_type],
    )

for row in disagreements.head(8).itertuples(index=False):
    label = f'{row.example_id}:{row.id_link}'
    axes[0].annotate(label, (row.upstream_continuity_abs_deviation, row.opposition_cos_trimmed_mean), xytext=(4, 4), textcoords='offset points', fontsize=8)
    axes[1].annotate(label, (row.length_over_raw_wet_width, row.width_model_probability), xytext=(4, 4), textcoords='offset points', fontsize=8)

axes[0].set(
    xlabel='Upstream continuity absolute deviation',
    ylabel='Trimmed mean opposition cosine',
    title='Dominant classification signals for disagreements',
)
axes[1].axhline(
    float(disagreements['width_model_threshold'].iloc[0]), color='black', linestyle='--', linewidth=1, label='Decision threshold'
)
axes[1].set(
    xlabel='Length / raw wet width',
    ylabel='Held-out correction probability',
    title='Probability and candidate-filter geometry',
)
axes[0].legend()
axes[1].legend()
fig.suptitle('Model disagreements; point size is proportional to √(raw wet width)', y=1.02)
fig.tight_layout()
plt.show()

### Pairwise disagreement matrices

In [ ]:
pairwise_review_matrix(
    disagreements, classification_columns,
    group_column='error_type', palette=error_colors, group_labels=error_labels,
    title='All classifier inputs against each other — model disagreements',
    decision_boundary=True, decision_reference_frame=dataset,
    decision_threshold=float(model_summary['candidate_models']['logistic_regression']['recommended_threshold']),
)

important_context_variables = [
    'raw_wet_width', 'raw_total_width', 'wet_fraction',
    'downstream_continuity_abs_deviation', 'mean_bank_chord_difference',
    'mean_relative_dt_difference', 'n_wet_threads_mean', 'link_length',
]
display(Markdown(
    'The context matrix has no decision boundary because these variables were not supplied to the classifier.'
))
pairwise_review_matrix(
    disagreements, important_context_variables,
    group_column='error_type', palette=error_colors, group_labels=error_labels,
    title='Important interpretation-only variables — model disagreements',
)

## Local geometry inspection for selected disagreements

In [ ]:
def plot_link_context(example_id: str, link_id: int, ax, title: str) -> None:
    example_id = str(example_id).zfill(2)
    links_path = RUNS_DIR / example_id / 'width_review' / 'reviewed_links_snapshot.gpkg'
    links = gpd.read_file(links_path)
    links['id_link'] = pd.to_numeric(links['id_link'], errors='coerce').astype('Int64')
    focal = links.loc[links['id_link'].eq(int(link_id))]
    if focal.empty:
        ax.text(0.5, 0.5, f'link {link_id} not found', ha='center', va='center')
        ax.set_axis_off()
        return
    geometry = focal.geometry.iloc[0]
    radius = max(float(geometry.length) * 2.0, 1.0)
    neighborhood = links.loc[links.geometry.intersects(geometry.buffer(radius))]
    neighborhood.plot(ax=ax, color='#B8B8B8', linewidth=1.2)
    focal.plot(ax=ax, color='#D62728', linewidth=3.0)
    minx, miny, maxx, maxy = geometry.buffer(radius).bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)
    ax.set_aspect('equal')
    ax.set_title(title)
    ax.set_axis_off()

inspection_rows = pd.concat([false_positives.head(2), false_negatives.head(2)], ignore_index=True)
if inspection_rows.empty:
    display(Markdown('No disagreements to map.'))
else:
    fig, axes = plt.subplots(1, len(inspection_rows), figsize=(5 * len(inspection_rows), 5))
    axes = np.atleast_1d(axes)
    for ax, row in zip(axes, inspection_rows.itertuples(index=False)):
        kind = 'false correction' if row.requires_correction == 0 else 'missed correction'
        plot_link_context(row.example_id, row.id_link, ax, f'{kind}\nexample {row.example_id}, link {row.id_link}\np={row.width_model_probability:.3f}')
    fig.tight_layout()
    plt.show()

## Provisional recommendation (review required)

In [ ]:
candidate = model_summary['candidate_models'][provisional_model]
display(Markdown(
    f"**Candidate:** `{provisional_model}` at threshold **{candidate['recommended_threshold']:.2f}**.  \\n"
    f"**Reason:** {model_summary['provisional_selection_reason']}.  \\n"
    f"**Held-out PR-AUC:** {candidate['metrics']['average_precision']:.3f}; "
    f"**split-sensitive false-correction rate:** {candidate['metrics']['split_sensitive_false_correction_rate']:.3%}.  \\n"
    f"**Production authorized:** `{model_summary['production_authorized']}`. "
    "Review the per-example failures, calibration, and mapped disagreements before approving a production threshold."
))